In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import random
import joblib
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
file_path = '/content/drive/MyDrive/Resume.csv'
df = pd.read_csv(file_path)


In [ ]:
df.isnull().sum()

,0
ID,0
Resume_str,0
Resume_html,0
Category,0


In [ ]:
df.drop("ID", axis=1, inplace=True)
df

,Resume_str,Resume_html,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR
...,...,...,...
2479,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2480,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION
2481,GEEK SQUAD AGENT Professional...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2482,PROGRAM DIRECTOR / OFFICE MANAGER ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION


In [ ]:
max_count = df['Category'].value_counts().max()
augmented_rows = []

In [ ]:
for category in df['Category'].unique():
    cat_df = df[df['Category'] == category]
    needed = max_count - len(cat_df)

    if needed > 0:
        existing_resumes = cat_df['Resume_str'].tolist()
        for i in range(needed):
            base_text = str(random.choice(existing_resumes))
            words = base_text.split()
            new_words = [w for w in words if random.random() > 0.05]
            augmented_rows.append({
                "ID": f"AUG_{category}_{i}",
                "Resume_str": " ".join(new_words),
                "Category": category
            })

if augmented_rows:
    df = pd.concat([df, pd.DataFrame(augmented_rows)], ignore_index=True)

df = df.sample(frac=1).reset_index(drop=True)
print(df['Category'].value_counts())

Category
TEACHER                   120
FITNESS                   120
DESIGNER                  120
AGRICULTURE               120
FINANCE                   120
APPAREL                   120
BUSINESS-DEVELOPMENT      120
ENGINEERING               120
ACCOUNTANT                120
HEALTHCARE                120
BANKING                   120
ARTS                      120
DIGITAL-MEDIA             120
PUBLIC-RELATIONS          120
CONSTRUCTION              120
SALES                     120
HR                        120
ADVOCATE                  120
AUTOMOBILE                120
INFORMATION-TECHNOLOGY    120
CONSULTANT                120
BPO                       120
CHEF                      120
AVIATION                  120
Name: count, dtype: int64


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'http\S+\s*', ' ', text)
    text = re.sub(r'RT|cc', ' ', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'@\S+', '  ', text)
    text = re.sub(r'[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]', ' ', text)
    text = re.sub(r'[^\x00-\x7f]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

df['Resume_str'] = df['Resume_str'].apply(clean_text)

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['Category'])
joblib.dump(le, 'label_encoder.pkl')

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Resume_str'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})

val_dataset = Dataset.from_dict({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'labels': val_labels
})

train_dataset.set_format('torch')
val_dataset.set_format('torch')

In [ ]:
num_labels = len(le.classes_)
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,              # 3 se badha kar 6 kar diya
    learning_rate=2e-5,              # Best learning rate for DistilBERT
    warmup_steps=100,                # Training smooth start hogi
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from transformers import TrainingArguments, Trainer

print("Setting up advanced configurations for 90%+ target...")

# 1. Define the new Training Arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,              # Increased to 6 epochs
    learning_rate=2e-5,              # Optimal learning rate
    warmup_steps=100,                # Smooth start for training
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

# 2. Re-initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# 3. Start Training
print("Training started! Please wait...")
trainer.train()

# 4. Save and Check Accuracy
print("\nSaving new high-accuracy model...")
model.save_pretrained('./distilbert_resume_model')
tokenizer.save_pretrained('./distilbert_resume_model')

print("\nCalculating Final Accuracy...")
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
acc = accuracy_score(val_labels, preds)

print(f"\n🏆 NEW TOTAL ACCURACY: {acc * 100:.2f}%\n")
print(classification_report(val_labels, preds, target_names=le.classes_))

Setting up advanced configurations for 90%+ target...
Training started! Please wait...


Epoch,Training Loss,Validation Loss
1,1.944343,1.928020
2,0.992395,1.017411
3,0.594439,0.714919
4,0.337583,0.657494
5,0.331632,0.611474
6,0.115913,0.613664


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Saving new high-accuracy model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Calculating Final Accuracy...



🏆 NEW TOTAL ACCURACY: 86.11%

                        precision    recall  f1-score   support

            ACCOUNTANT       0.91      0.95      0.93        21
              ADVOCATE       0.78      0.82      0.80        22
           AGRICULTURE       0.75      0.84      0.79        25
               APPAREL       0.68      0.63      0.65        27
                  ARTS       0.79      0.48      0.59        23
            AUTOMOBILE       0.77      0.75      0.76        36
              AVIATION       0.83      0.83      0.83        23
               BANKING       0.78      0.70      0.74        20
                   BPO       0.94      0.97      0.95        32
  BUSINESS-DEVELOPMENT       1.00      1.00      1.00        17
                  CHEF       1.00      0.76      0.86        21
          CONSTRUCTION       1.00      0.97      0.98        29
            CONSULTANT       0.86      0.93      0.89        27
              DESIGNER       0.95      1.00      0.98        21
        